# Dataset Raw de xG - EDA

> **Múltiples ligas**&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuente:** Understat (vía soccerdata)

## Objetivos

- Verificar cobertura temporal y por liga (2014-15 – 2023-24, 3 ligas).
- Identificar variables disponibles y tipos de datos.
- Validar integridad: duplicados, nulos, negativos, drift y outliers de xG.
- Comparar nomenclatura de equipos, ligas, temporadas y fechas con el core clean.
- Exportar dataset xG raw en Parquet.

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---------|----------|
| 0 | Entorno y configuración | Librerías, rutas y datasets de referencia |
| 1 | Extracción de datos | Descarga de xG desde Understat vía soccerdata |
| 2 | Inspección inicial | Dimensiones, índice y tipos de dato |
| 3 | Cobertura temporal | Partidos por liga y temporada |
| 4 | Validaciones de integridad | Duplicados, nulos, negativos y outliers |
| 5 | Compatibilidad con football-data | Equipos, fechas, ligas y temporadas |
| 6 | Conclusiones | Hallazgos y decisiones para el merge |
| 7 | Exportación | Guardado del dataset xG en Parquet |


---
##

## 0) Entorno y configuración

Configuración de dependencias, rutas del proyecto y parámetros de extracción.

In [ ]:
from pathlib import Path
import pandas as pd
import soccerdata as sd
import sys
import json
from IPython.display import display

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if any((p / m).exists() for m in {"config", "src", "data"})
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
RAW_XG_ROOT = PROJECT_ROOT / "data" / "processed" / "xg"
XG_RAW_PATH = RAW_XG_ROOT / "xg_validated.parquet"
XG_VALIDATED_SCHEMA_PATH = RAW_XG_ROOT / "xg_validated_schema.json"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CORE_CLEAN_PATH = PROCESSED_ROOT / "multi_league" / "core_multi_league_clean.parquet"

# Importación de funciones propias
from src.analysis import build_team_mapping
from src.utils import (
    get_duplicates_count,
    get_negatives_dict,
    get_nulls_dict,
    get_drift_columns,
    get_false_conditions,
    get_outliers_count,
    get_goals_cross_check,
)

### 0.1 Configuración de referencia

In [ ]:
with open(CONFIG_ROOT / "leagues.json") as f:
    ALL_LEAGUES = json.load(f)

with open(PROCESSED_ROOT / "multi_league" / "core_multi_league_clean_schema.json") as f:
    schema = json.load(f)

with open(CONFIG_ROOT / "league_mapping.json") as f:
    LEAGUE_MAP = json.load(f)

df_core = pd.read_parquet(CORE_CLEAN_PATH)
print(f"Core clean cargado: {len(df_core):,} partidos × {len(df_core.columns)} columnas")

---
##

## 1) Extracción de datos

Descarga de estadísticas de xG a nivel de partido desde `Understat` vía `soccerdata`.

In [ ]:
SEASONS = [f"{y}{y+1}" for y in range(14, 24)]
# → ['1415', '1516', ..., '2324'] --> Formato interno de Soccerdata

us = sd.Understat(leagues=list(LEAGUE_MAP.keys()), seasons=SEASONS)
df_xg = us.read_schedule()

print(f"Dataset xG extraído: {df_xg.shape[0]:,} partidos × {df_xg.shape[1]} variables")
print(f"Temporadas: 20{SEASONS[0][:2]}-20{SEASONS[0][2:]} – 20{SEASONS[-1][:2]}-20{SEASONS[-1][2:]}")


---
##

## 2) Inspección inicial

Dimensiones, tipos de datos y vista de las primeras filas.

### 2.1 Resumen estructural del dataset

In [ ]:
print(f"Dimensiones: {df_xg.shape[0]:,} partidos × {df_xg.shape[1]} variables")
print(f"Índice: {df_xg.index.names} ({df_xg.index.nlevels} niveles)")
print(f"Ligas: {df_xg.index.get_level_values('league').unique().tolist()}")
print(f"Temporadas: {df_xg.index.get_level_values('season').nunique()}")

### 2.2 Tipos de datos

In [ ]:
types = pd.DataFrame({"Variable": df_xg.dtypes.index, "Tipo": df_xg.dtypes.values})
display(types.style.hide(axis="index").set_table_attributes('style="max-height:300px; overflow-y:auto; display:block"'))


### 2.3 Preview del dataset

In [ ]:
display(df_xg.head(5).style.format({"home_xg": "{:.2f}", "away_xg": "{:.2f}"}).hide(axis="index"))

---
##

## 3) Cobertura temporal

Verificación del número de partidos por liga y temporada frente a los valores esperados del core clean.

In [ ]:
n_seasons = df_xg.index.get_level_values("season").nunique()
expected = {us: schema["matches_per_league"][core] // n_seasons for us, core in LEAGUE_MAP.items()}

coverage = df_xg.groupby(level=["league", "season"]).size().unstack(fill_value=0)
coverage.columns = [f"{s[:2]}/{s[2:]}" for s in coverage.columns]
coverage.index.name = None
display(coverage)

for lg, exp in expected.items():
    gaps = coverage.loc[lg][coverage.loc[lg] != exp]
    status = "⚠" if len(gaps) else "✓"
    print(f"{status} {lg:<20} {exp}/temporada" + (f" — gaps: {gaps.to_dict()}" if len(gaps) else ""))

---
##

## 4) Validaciones de integridad

Verificación de integridad del dataset: duplicados, nulos, drift de tipos entre temporadas, valores negativos, outliers de xG, partidos sin resultado y validación cruzada de goles contra el core clean.

In [ ]:
print("── Resumen integridad ─────────────────────────────────────────")
summary = {
    "Duplicados": get_duplicates_count(df_xg),
    "Negativos": get_negatives_dict(df_xg),
    "Nulos": get_nulls_dict(df_xg),
    "Drift de tipos": get_drift_columns(df_xg, "season"),
    "Sin resultado": get_false_conditions(df_xg, "is_result"),
    "xG > 10": get_outliers_count(df_xg, ["home_xg", "away_xg"], 10.0),
}
for metric, value in summary.items():
    v = value if value else "ninguno"
    print(f"  {metric:<18} {v}")
print("\n ── Goles xG vs core ──────────────────────────────────────")
cross_check = get_goals_cross_check(df_xg, df_core, LEAGUE_MAP)
for lg_us, (match, g_xg, g_core) in cross_check.items():
    print(f"  [{match}] {lg_us:<25} xG={g_xg:>6,}  core={g_core:>6,}")
print("───────────────────────────────────────────────────────")

---
##

## 5) Compatibilidad con `football-data`

Comparación de nombres de equipos, ligas, temporadas y formato de fechas entre `Understat` y el core clean. 

### 5.1 Nombres de equipos

Equipos únicos por liga en cada fuente y diferencias de nomenclatura.

In [ ]:
rows = []
for lg_us, lg_core in LEAGUE_MAP.items():
    teams_xg   = set(df_xg.loc[lg_us]["home_team"].unique())
    teams_core = set(df_core[df_core["League"] == lg_core]["HomeTeam"].unique())
    only_xg    = sorted(teams_xg - teams_core)
    only_core  = sorted(teams_core - teams_xg)
    
    print(f"\n── {lg_us}")
    print(f"  xG={len(teams_xg)}  core={len(teams_core)}  coinciden={len(teams_xg & teams_core)}")
    if only_xg:
        print(f"  solo xG   → {only_xg}")
    if only_core:
        print(f"  solo core → {only_core}")

#### 5.1.1 Generación del mapping de equipos

Mapping automático de nombres de equipos entre `Understat` y `football-data` mediante fuzzy matching en tres rondas.

In [ ]:
TEAM_MAP = build_team_mapping(df_xg, df_core, LEAGUE_MAP)

mapping_path = CONFIG_ROOT / "team_mapping_xg.json"
with open(mapping_path, "w") as f:
    json.dump(TEAM_MAP, f, indent=2, ensure_ascii=False)

print(f"Team mapping: {len(TEAM_MAP)} equipos → {mapping_path.relative_to(PROJECT_ROOT)}\n")
for us, core in list(sorted(TEAM_MAP.items()))[:5]:
    print(f"  {us:<20} →   {core:<20}")
print(f"  ...")

### 5.2 Fechas

Tipo de dato y ejemplo en ambas fuentes.

In [ ]:
print(f"  {'Fuente':<12} {'Tipo':<20} {'Ejemplo'}")
print(f"{'━'*50}")
print(f"  {'xG':<12} {str(df_xg['date'].dtype):<20} {df_xg['date'].iloc[0]}")
print(f"  {'core':<12} {str(df_core['Date'].dtype):<20} {df_core['Date'].iloc[0].strftime('%Y-%m-%d')}")

#### 5.2.1 Desfase de fechas por huso horario


Identificación de partidos donde la fecha difiere entre Understat y football-data.

In [ ]:
df_xg_flat = df_xg.reset_index()

k_xg = ("20" + df_xg_flat["season"].str[2:]) + "_" + \
       df_xg_flat["league"].map(LEAGUE_MAP) + "_" + \
       df_xg_flat["home_team"].map(TEAM_MAP).fillna(df_xg_flat["home_team"]) + "_" + \
       df_xg_flat["away_team"].map(TEAM_MAP).fillna(df_xg_flat["away_team"])

k_core = df_core["Season"].astype(str) + "_" + df_core["League"] + "_" + \
         df_core["HomeTeam"] + "_" + df_core["AwayTeam"]

df_test = pd.DataFrame({"xG": df_xg_flat["date"].dt.normalize().dt.date.values}, index=k_xg)
df_test = df_test.join(pd.Series(df_core["Date"].dt.date.values, index=k_core, name="core"), how="inner")
date_mismatch = df_test[df_test["xG"] != df_test["core"]].dropna()

print(f"  Partidos con desfase de fecha: {len(date_mismatch)} de {len(df_test):,}\n")
date_mismatch_display = date_mismatch.head(5).reset_index()
date_mismatch_display.columns = ["Muestra de partidos inconsistentes", "xG", "core"]

display(date_mismatch_display.style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th, td", "props": [("text-align", "center")]},
        {"selector": "th:first-child, td:first-child", "props": [("text-align", "left")]},
    ])
)

> **Nota de implementación** — Inicialmente se consideró incluir la fecha como componente del `match_id` para garantizar unicidad. Sin embargo, el análisis revela que 95 partidos (0,9 % del dataset) presentan un desfase de ±1 día entre fuentes, atribuible a diferencias de huso horario en encuentros nocturnos — Understat registra en UTC, football-data en hora local. La clave `league + season + home_team + away_team` es igualmente unívoca dado que un equipo solo ejerce de local una vez frente a cada rival en la misma temporada, eliminando esta fuente de inconsistencia.

### 5.3 Ligas y temporadas

Nomenclatura de ligas y formato de temporadas.

In [ ]:
leagues_xg   = sorted(df_xg.index.get_level_values("league").unique())
leagues_core = sorted(df_core["League"].unique())
seasons_xg   = sorted(df_xg.index.get_level_values("season").unique())
seasons_core = sorted(df_core["Season"].unique())

print(f"  Ligas xG          {leagues_xg}")
print(f"  Ligas core        {leagues_core}")
print()
print(f"  Temporadas xG     {seasons_xg}")
print(f"  Temporadas core   {seasons_core}")


---
##

## 6) Conclusiones del análisis exploratorio

### Cobertura

El dataset contiene **10.660 partidos × 17 variables**, con cobertura temporal idéntica al core clean:  
**380 partidos por temporada** en Premier League y La Liga y **306 en Bundesliga**, sin gaps en ninguna temporada.

### Integridad

No se detectaron **duplicados, nulos, valores negativos, drift de tipos ni outliers de xG**.  
Los goles (`home_goals`, `away_goals`) coinciden exactamente con el core dataset en las tres ligas.

### Variables útiles

Para el merge se incorporarán únicamente:

- **`home_xg`**
- **`away_xg`**

Las variables `home_goals` y `away_goals` se utilizan únicamente para validar el join.  
El resto corresponde a metadata de `Understat` y no se integrará en el dataset final.

### Compatibilidad con el core dataset

Se identifican algunas diferencias estructurales entre ambos datasets que deben resolverse antes del merge:

| Aspecto | Understat | Core clean | Acción |
|---|---|---|---|
| Equipos | Nombres completos | Algunos abreviados | Aplicar diccionario de mapping |
| Ligas | `ENG-Premier League` (ej.) | `premier` | Aplicar mapping |
| Temporadas | `1415` (ej.) | `2015` | Mapping en merge |
| `match_id` | — | `League_Season_Home_Away` | Construir tras aplicar mappings |

### Preparación del merge

El join se realizará mediante **`match_id`**, construido en el dataset xG tras aplicar los mappings de **equipos, ligas y temporadas**. Dado que la cobertura de partidos es idéntica al core dataset (**10.660 partidos**), se espera **coincidencia completa**. Cualquier discrepancia indicará que alguna variable no fue mapeada correctamente.

---
##

## 7) Exportación del raw xG dataset 

Guardado del dataset xG raw en Parquet para consumo en el notebook de merge.

In [ ]:
RAW_XG_ROOT.mkdir(parents=True, exist_ok=True)
df_xg_export = df_xg.reset_index()

schema_xg = {
    "num_rows": len(df_xg_export),
    "num_columns": len(df_xg_export.columns),
    "columns": {col: str(df_xg_export[col].dtype) for col in df_xg_export.columns},
    "matches_per_league": df_xg_export.groupby("league").size().to_dict(),
    "seasons": sorted(df_xg_export["season"].unique().tolist()),
    "source": "understat (vía soccerdata)"
}

df_xg_export.to_parquet(XG_RAW_PATH, index=False)

with open(XG_VALIDATED_SCHEMA_PATH, "w") as f:
    json.dump(schema_xg, f, indent=2, ensure_ascii=False)

print(f"Dataset xG exportado:")
print(f"  {len(df_xg_export):,} filas × {len(df_xg_export.columns)} columnas\n")

print(f"Archivos guardados:")
print(f"  · Dataset → {XG_RAW_PATH.relative_to(PROJECT_ROOT)}")
print(f"  · Esquema → {XG_VALIDATED_SCHEMA_PATH.relative_to(PROJECT_ROOT)}")

---
##